In [0]:
#pip install -U --quiet databricks-langchain databricks-agents mlflow databricks-vectorsearch langchain langchain-core PyMuPDF
dbutils.library.restartPython()


In [0]:
import mlflow
import fitz
from operator import itemgetter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [0]:
# Configuration centralized
chain_config = {
    "volume_path": "/Volumes/db_workspace_claire/default/these/2025-09-eb-big-book-of-machine-learning-3rd-ed.pdf",
    "delta_table_name": "db_workspace_claire.default.chunked_data",
    "vector_search_endpoint": "vector_search_endpoint_name",
    "vector_index_path": "db_workspace_claire.default.big_book_of_machine_learning_index",
    "embedding_endpoint": "bge_base_en_v1_5",
    "llm_endpoint": "databricks-meta-llama-3.1-405b-instruct",
    "llm_prompt_template": """You are an assistant that answers questions based on a specific document. 
    Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say you don't know.
    
    Context: {context}"""
}

In [0]:
chain_config

In [0]:
# --- Extraction & Chunking ---
def extract_and_chunk(path):
    doc = fitz.open(path)
    text = "".join([page.get_text() for page in doc])
    doc.close()
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_text(text)
    
    return [{"source": path, "chunk_id": i, "content": chunk} for i, chunk in enumerate(chunks)]

data = extract_and_chunk(chain_config.get("volume_path"))

In [0]:
# --- Save to Delta ---
spark.createDataFrame(data).write.mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(chain_config.get("delta_table_name"))

In [0]:
# --- Vector Search Setup ---
vsc = VectorSearchClient()

# Ensure endpoint exists

try:
    vsc.get_endpoint(name=chain_config.get("vector_search_endpoint"))
    print(f" Endpoint '{chain_config.get('vector_search_endpoint')}' is already there.")
except:
    print(f"Creating endpoint '{chain_config.get('vector_search_endpoint')}'... This takes ~10 mins.")
    vsc.create_endpoint(name=chain_config.get("vector_search_endpoint"), endpoint_type="STANDARD")

In [0]:
vsc = VectorSearchClient()

# Ensure index exists
try:
    index = vsc.get_index(chain_config.get("vector_search_endpoint"), chain_config.get("vector_index_path"))
except:
    index = vsc.create_delta_sync_index(
        endpoint_name=chain_config.get("vector_search_endpoint"),
        source_table_name=chain_config.get("delta_table_name"),
        index_name=chain_config.get("vector_index_path"),
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=chain_config.get("embedding_endpoint")
    )
    

In [0]:
mlflow.end_run()

In [0]:
import mlflow
import yaml

# 1. Clean up active runs and set experiment
mlflow.end_run()
user_email = spark.sql('SELECT current_user()').collect()[0][0]
mlflow.set_experiment(f"/Users/{user_email}/rag_ml_book")

# 2. Explicitly create the YAML file on the local disk first
# This ensures MLflow has a physical file to grab
with open("model_config.yaml", "w") as f:
    yaml.dump(chain_config, f)

# 3. Define production-ready dependencies
required_libraries = [
    "databricks-langchain",
    "databricks-vectorsearch",
    "langchain",
    "langchain-core",
    "PyMuPDF",
    "pyyaml"
]

with mlflow.start_run():
    # Log parameters for UI visibility
    mlflow.log_params(chain_config)
    
    signature = mlflow.models.infer_signature(
        model_input={"question": "What is machine learning?"},
        model_output="Machine learning is..."
    )
    
    # 4. Log model using 'code_paths'
    # This is the "Pro" way to ensure 'model_config.yaml' is packaged 
    # right next to 'chain_driver.py' in the container.
    model_info = mlflow.langchain.log_model(
        lc_model="chain_driver.py",
        code_paths=["model_config.yaml"], 
        artifact_path="chain",
        registered_model_name="db_workspace_claire.default.rag_big_book_ml",
        signature=signature,
        pip_requirements=required_libraries
    )

version_number = model_info.model_uri.split('/')[-1]
print(f"✅ Success! Deploy Version {version_number} to your endpoint.")

In [0]:
import mlflow

# Replace with the version number you just created (e.g., "5")
model_version = version_number 
model_uri = f"models:/db_workspace_claire.default.rag_big_book_ml/7"

print(f"Checking Model URI: {model_uri}")

try:
    # 1. Load the model as a generic Python function
    # This triggers the 'chain_driver.py' logic and loads the YAML
    loaded_model = mlflow.pyfunc.load_model(model_uri)
    
    # 2. Test a real prediction
    test_input = {"question": "What is the Big Book of Machine Learning about?"}
    response = loaded_model.predict(test_input)
    
    print("\n✅ TEST PASSED: The model loaded and responded successfully.")
    print(f"Response: {response}")

except Exception as e:
    print("\n❌ TEST FAILED: There is still an issue in the configuration or code.")
    print(f"Error Details: {e}")

In [0]:
loaded_model = mlflow.pyfunc.load_model(f"models:/db_workspace_claire.default.rag_big_book_ml/{model_info.version}")
print(loaded_model.predict({"question": "Qu'est-ce que le ML ?"}))